# PreTest 04: Unsupervised Learning — การแบ่งกลุ่มลูกค้า (Customer Segmentation)
### K-Means · Hierarchical Clustering · DBSCAN · PCA · Elbow / Silhouette / Davies-Bouldin

> **ที่มา:** ดัดแปลงและขยายจาก Kaggle Notebook
> [Customer Segmentation (K-Means) | Analysis](https://www.kaggle.com/code/kushal1996/customer-segmentation-k-means-analysis) โดย Kushal
> สิ่งที่เพิ่ม: คำอธิบายภาษาไทย, Scaling, Silhouette/Davies-Bouldin, Hierarchical, DBSCAN, PCA, การตีความเป็น Persona
>
> **ชุดข้อมูล (Dataset):** [Mall Customer Segmentation Data](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) — ลูกค้าห้าง 200 คน

---

## Supervised Learning เทียบกับ Unsupervised Learning

| | Supervised (PreTest 01–03) | **Unsupervised (บทนี้)** |
|---|---|---|
| มีคำตอบ (y) หรือไม่ | มี | **ไม่มี** |
| เป้าหมาย | ทำนายคำตอบ | **หาโครงสร้าง/กลุ่มที่ซ่อนอยู่** |
| Train/Test Split | ต้องมี | ไม่จำเป็น (ไม่มีคำตอบให้เทียบ) |
| วัดผลด้วย | Accuracy, F1, R² ... | **Inertia, Silhouette, Davies-Bouldin** + **ความหมายทางธุรกิจ** |

## ขั้นตอน
1 โจทย์ → 2 โหลด → 3 สำรวจ → 4 Cleaning → 5 EDA → 6 เลือก Feature + Scale → 7 K-Means → 8 Hierarchical → 9 DBSCAN → 10 เปรียบเทียบ → 11 PCA (3 Feature) → 12 Persona และข้อเสนอแนะ

---
# ขั้นที่ 1: กำหนดโจทย์
- **เป้าหมาย:** แบ่งลูกค้าเป็นกลุ่มที่มีพฤติกรรมคล้ายกัน เพื่อให้ฝ่ายการตลาดทำโปรโมชันเฉพาะกลุ่ม
- **ประเภทปัญหา:** ไม่มีคอลัมน์คำตอบ → **Clustering (Unsupervised)**
- **วัดผลด้วย:**

| Metric | ความหมาย | ดีเมื่อ |
|---|---|---|
| **Inertia** (WCSS) | ผลรวมระยะ² จากจุดถึงจุดศูนย์กลางกลุ่ม (กลุ่มแน่นเพียงใด) | ต่ำ (แต่ลดลงเสมอเมื่อเพิ่ม k → ใช้ Elbow) |
| **Silhouette Score** | จุดอยู่ใกล้กลุ่มตัวเองมากกว่ากลุ่มอื่นเพียงใด (−1 ถึง 1) | **สูง** (> 0.5 ถือว่าดี) |
| **Davies-Bouldin Index** | ความคล้ายระหว่างกลุ่ม (กลุ่มแยกกันชัดเพียงใด) | **ต่ำ** |

# ขั้นที่ 2: นำเข้าไลบรารีและโหลดข้อมูล

---
# ขั้นที่ 3: สำรวจข้อมูลเบื้องต้น

| คอลัมน์ | ความหมาย |
|---|---|
| `CustomerID` | รหัสลูกค้า (ไม่ใช้ในการจัดกลุ่ม) |
| `Gender` | เพศ |
| `Age` | อายุ |
| `Annual Income (k$)` | รายได้ต่อปี (พันดอลลาร์) |
| `Spending Score (1-100)` | คะแนนการใช้จ่ายที่ห้างให้ (ยิ่งสูง ยิ่งใช้จ่ายมาก/บ่อย) |

---
# ขั้นที่ 4: ทำความสะอาดข้อมูล

> **ผลลัพธ์:** ข้อมูลสะอาด ไม่มีค่าหาย ไม่มีแถวซ้ำ — ลบ `CustomerID` เพราะหากนำมาใช้ โมเดลจะจัดกลุ่มตามเลขรหัส ซึ่งไม่มีความหมาย

---
# ขั้นที่ 5: EDA & Visualization

### 5.1 การกระจายของแต่ละตัวแปร

### 5.2 ความสัมพันธ์ระหว่างตัวแปร

> **Insight:** กราฟ Income vs Spending สามารถสังเกตเห็นได้ **5 กลุ่ม** (มุมทั้ง 4 + ตรงกลาง)
> - Age vs Spending: คนอายุ > 40 แทบไม่มีใครใช้จ่ายสูงมาก
> - Gender กระจายปนกันทุกกราฟ → **เพศไม่ช่วยแบ่งกลุ่ม** จึงไม่ใช้

---
# ขั้นที่ 6: เลือก Feature และ Scale

- **Feature หลัก:** `Income` + `Spending` (2 มิติ → วาดกราฟดูผลได้ง่าย ตรงกับโจทย์การตลาด)
- **ต้อง Scale:** ทุกวิธีในบทนี้ใช้ **ระยะทาง** ถ้าตัวแปรมีช่วงต่างกัน ตัวที่ช่วงกว้างจะครอบงำ

---
# ขั้นที่ 7: โมเดลที่ 1 — K-Means

### หลักการ
1. สุ่มวางจุดศูนย์กลาง (Centroid) k จุด
2. ให้ทุกจุดข้อมูลเข้ากลุ่มของ Centroid ที่ใกล้ที่สุด
3. ย้าย Centroid ไปที่ค่าเฉลี่ยของสมาชิกในกลุ่ม
4. ทำข้อ 2–3 ซ้ำจนไม่เปลี่ยน

**ต้องกำหนด k เอง** → ใช้ Elbow + Silhouette ช่วยตัดสินใจ

### 7.1 เลือกจำนวนกลุ่ม (k)

> **Elbow:** Inertia ลดลงเร็วจนถึง k = 5 แล้วจึงลดช้าลง (ข้อศอกอยู่ที่ 5)
> **Silhouette สูงสุด** และ **Davies-Bouldin ต่ำสุด** ที่ k = 5 เช่นกัน → ทั้ง 3 วิธีตรงกัน และสอดคล้องกับที่สังเกตได้จากกราฟในขั้นที่ 5 → เลือก **k = 5**

### 7.2 เทรน K-Means (k = 5) และดูผล

### 7.3 Profile ของแต่ละกลุ่ม

### 7.4 การวิเคราะห์ผล K-Means

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| ควรแบ่งกี่กลุ่ม? | Elbow, Silhouette, Davies-Bouldin ชี้ **k = 5** ตรงกันทั้ง 3 วิธี | มั่นใจได้ และสอดคล้องกับที่สังเกตได้จากกราฟ 5.2 |
| กลุ่มแยกชัดหรือไม่ | Silhouette **0.555** (> 0.5) | ดี — ลูกค้าแต่ละคนใกล้กลุ่มตัวเองมากกว่ากลุ่มอื่นชัดเจน |
| กลุ่มปนกันหรือไม่ | Davies-Bouldin **0.572** (ต่ำสุดในทุก k) | กลุ่มแยกกันดี |
| แต่ละกลุ่มเป็นอย่างไร? | 5 กลุ่ม = มุมทั้ง 4 ของกราฟ (รายได้ต่ำ/สูง × ใช้จ่ายต่ำ/สูง) + กลุ่มกลาง | กลุ่มกลางใหญ่ที่สุด **81 คน (40%)** |
| เพศมีผลหรือไม่ | % ผู้หญิงในแต่ละกลุ่ม 46–61% | ใกล้สัดส่วนรวม (56%) → เพศไม่ใช่ตัวแบ่งกลุ่ม |

> **สรุป:** K-Means แบ่งได้ 5 กลุ่มที่ชัดเจนและอธิบายได้ทางธุรกิจ (ตั้งชื่อ Persona ในขั้นที่ 12)

---
# ขั้นที่ 8: โมเดลที่ 2 — Hierarchical Clustering (Agglomerative)

### หลักการ
เริ่มจาก **ทุกจุดเป็นกลุ่มของตัวเอง** แล้วรวมคู่ที่ใกล้ที่สุดทีละคู่ จนเหลือกลุ่มเดียว
→ วาดเป็น **Dendrogram** (แผนผังต้นไม้) แล้ว "ตัด" ที่ความสูงที่เหมาะสมเพื่อได้จำนวนกลุ่ม
- `linkage='ward'` = รวมคู่ที่ทำให้ความแปรปรวนภายในกลุ่มเพิ่มน้อยที่สุด (นิยมที่สุด)
- **ข้อดี:** ไม่ต้องกำหนด k ล่วงหน้า ดูจาก Dendrogram ได้

### 8.1 Dendrogram

> **วิธีอ่าน:** แกนตั้งคือระยะที่กลุ่มถูกรวมกัน — หาช่วงที่ **เส้นแนวตั้งยาวที่สุดโดยไม่มีการรวมกลุ่มเกิดขึ้น** แล้วตัดในช่วงนั้น
> ช่วงระยะ ~4.4 ถึง ~9.5 ไม่มีการรวมกลุ่มเกิดขึ้น → ตัดที่ ≈ 6 (เส้นแดง) ตัดผ่านเส้นแนวตั้ง **5 เส้น → 5 กลุ่ม** ตรงกับ K-Means
> (ข้อควรระวัง: หากตัดที่ระดับ 4 จะได้ 8 กลุ่ม)

### 8.2 เทรนและเทียบกับ K-Means

### 8.3 การวิเคราะห์ผล Hierarchical

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| Dendrogram บอกกี่กลุ่ม? | ช่วงว่างยาวที่สุดอยู่ระหว่างระยะ ~4.4–9.5 → **5 กลุ่ม** | ยืนยัน k = 5 โดยไม่ต้องใช้ Elbow |
| คุณภาพกลุ่ม | Silhouette **0.554**, Davies-Bouldin **0.578** | แทบเท่า K-Means (0.555 / 0.572) |
| ได้กลุ่มเหมือน K-Means หรือไม่ | ตรงกัน **196 / 200 คน** (98%) | สองวิธีต่างหลักการแต่ได้คำตอบเดียวกัน → ผลน่าเชื่อถือ |
| 4 คนที่ต่างคือใคร? | ลูกค้าที่อยู่ **รอยต่อ** ระหว่างกลุ่มกลางกับกลุ่มข้างเคียง | เป็นลักษณะปกติของข้อมูลที่อยู่บริเวณขอบกลุ่ม |

> **สรุป:** ได้ผลเกือบเหมือน K-Means — ข้อดีคือมี Dendrogram ให้เห็นว่ากลุ่มไหนคล้ายกันและควรแบ่งกี่กลุ่ม

---
# ขั้นที่ 9: โมเดลที่ 3 — DBSCAN

### หลักการ
หากลุ่มจาก **ความหนาแน่น** — จุดที่มีเพื่อนบ้านอย่างน้อย `min_samples` ตัวในรัศมี `eps` คือแกนกลุ่ม
- **ไม่ต้องกำหนดจำนวนกลุ่ม** — ได้จำนวนกลุ่มเอง
- จุดที่ไม่อยู่ในกลุ่มใดจะถูกติดป้าย **`-1` = Noise (ค่าผิดปกติ)** ← ใช้หา Outlier ได้
- สามารถจับกลุ่มที่มีรูปร่างไม่เป็นทรงกลมได้ ซึ่งต่างจาก K-Means

### 9.1 เลือก `eps` ด้วย k-distance plot
เรียงระยะถึงเพื่อนบ้านลำดับที่ `min_samples` ของทุกจุด แล้วหา "ข้อศอก" ของกราฟ

> **ข้อสังเกต:** กราฟ k-distance เริ่มชันขึ้นช่วง ~0.35–0.5 (ข้อศอก) / `eps` เล็ก → ได้จำนวนกลุ่มและ Noise มาก / `eps` ใหญ่ → กลุ่มรวมกันจนเหลือน้อย
> ค่า **0.35** สมดุลที่สุด (Silhouette สูงสุด, Noise ไม่มากเกินไป)

### 9.2 เทรนและดูผล

### 9.3 การวิเคราะห์ผล DBSCAN

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| ได้กี่กลุ่ม? | **6 กลุ่ม + Noise 23 คน** (11.5%) | ไม่ต้องกำหนดจำนวนกลุ่มเอง |
| ต่างจาก K-Means อย่างไร? | แยกกลุ่ม "รายได้ต่ำ" ออกเป็น 3 กลุ่มย่อย (ใช้จ่ายสูง 16, ต่ำ 12, ปานกลาง 7 คน) | กลุ่มรายได้ต่ำ-ใช้จ่ายปานกลางมีเพียง 7 คน อาจเล็กเกินไปที่จะทำการตลาดแยก |
| Noise คือใคร? | 13 จาก 23 คน มีรายได้ ≥ $100k (รายได้สูงสุดในห้าง) ที่เหลืออยู่ขอบกลุ่ม | ใช้หา **ลูกค้าผิดปกติ (Outlier)** ได้ — เช่นกลุ่มรายได้สูงมากที่อาจต้องดูแลพิเศษ |
| คุณภาพกลุ่ม | Silhouette **0.558**, Davies-Bouldin **0.511** | สูงที่สุด แต่ **คำนวณโดยตัด Noise ออก** (ไม่นับจุดที่จัดกลุ่มได้ยาก) → เทียบกับ K-Means ไม่ยุติธรรม |

> **สรุป:** DBSCAN เหมาะกับข้อมูลที่กลุ่มมีรูปร่างแปลก หรือต้องการหา Outlier
> ข้อมูลชุดนี้กลุ่มเป็นก้อนกลมชัดเจน K-Means จึงเหมาะกว่า แต่ DBSCAN ให้ข้อมูลเพิ่มเรื่อง **ลูกค้าที่ไม่เข้ากลุ่มใด**

---
# ขั้นที่ 10: เปรียบเทียบ 3 วิธี

### 10.1 การวิเคราะห์ผลการเปรียบเทียบ

| วิธี | กลุ่ม | Noise | Silhouette | Davies-Bouldin | ข้อดี | ข้อจำกัด |
|---|---|---|---|---|---|---|
| **K-Means** | 5 | 0 | 0.555 | 0.572 | เร็ว ง่าย มี Centroid ไว้ตีความ | ต้องเลือก k เอง, ชอบกลุ่มทรงกลม |
| Hierarchical | 5 | 0 | 0.554 | 0.578 | มี Dendrogram ช่วยเลือกจำนวนกลุ่ม | ช้าเมื่อข้อมูลใหญ่ |
| DBSCAN | 6 | 23 | 0.558* | 0.511* | หา Outlier ได้ ไม่ต้องกำหนด k | เลือก eps ยาก, *คะแนนไม่รวม Noise |

**ข้อสรุป**
1. K-Means และ Hierarchical **ได้กลุ่มเกือบเหมือนกัน (98%)** → กลุ่มลูกค้า 5 กลุ่มนี้ "มีอยู่จริง" ไม่ได้ขึ้นกับอัลกอริทึม
2. ค่า Silhouette ทั้งสามใกล้กันมาก (0.554–0.558) → ไม่สามารถตัดสินจากตัวเลขเพียงอย่างเดียวได้ ต้องดู **การใช้งานจริง**
3. **เลือก K-Means** เพราะลูกค้าทุกคนได้กลุ่ม (ไม่มี Noise) ทีมการตลาดใช้งานง่าย และ Centroid อธิบายแต่ละกลุ่มได้ทันที
4. ใช้ DBSCAN เสริมเพื่อดู **ลูกค้า 23 คนที่ผิดปกติ**

---
# ขั้นที่ 11 (เพิ่มเติม): ใช้ 3 Feature + PCA เพื่อวาดกราฟ

ถ้าใช้ `Age` + `Income` + `Spending` (3 มิติ) จะไม่สามารถแสดงเป็นกราฟ 2 มิติได้โดยตรง
→ ใช้ **PCA (Principal Component Analysis)** ย่อ 3 มิติเหลือ 2 มิติ โดยเก็บข้อมูล (ความแปรปรวน) ไว้มากที่สุด

### 11.1 การวิเคราะห์ผล PCA + 3 Feature

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| k ที่ดีที่สุด | **k = 6** (Silhouette 0.428) | เพิ่ม Age แล้วได้กลุ่มเพิ่ม 1 กลุ่ม |
| คุณภาพกลุ่ม | Silhouette 0.428 < 0.555 (2 Feature) | กลุ่มแยกกันชัดน้อยลง — เพิ่ม Feature ไม่ได้ทำให้ดีขึ้นเสมอ |
| PCA เก็บข้อมูลได้เพียงใด? | PC1 44% + PC2 33% = **78%** | กราฟ 2 มิติแสดงข้อมูล 3 มิติได้ค่อนข้างครบ |
| กลุ่มที่เพิ่มมาคืออะไร? | กลุ่มกลาง (รายได้/ใช้จ่ายปานกลาง) ถูกแยกตามอายุ: **วัยทำงานตอนต้น ~27 ปี** กับ **วัยกลางคน ~56 ปี** | Age ช่วยแยกกลุ่มกลางที่ใหญ่ที่สุดให้ละเอียดขึ้น |

> **สรุป:** ถ้าทีมการตลาดอยากทำแคมเปญแยกตามวัยสำหรับกลุ่มกลาง ใช้แบบ 3 Feature (6 กลุ่ม)
> ถ้าต้องการกลุ่มที่ชัดและง่าย ใช้แบบ 2 Feature (5 กลุ่ม)

---
# ขั้นที่ 12: Persona และข้อเสนอแนะทางธุรกิจ

ตั้งชื่อกลุ่มจาก **ค่าเฉลี่ยรายได้และการใช้จ่าย** ของแต่ละกลุ่ม (ใช้โค้ดตั้งชื่อ เพราะหมายเลขกลุ่มอาจสลับเมื่อรันบนเครื่อง/เวอร์ชันอื่น)

### 12.1 Persona และข้อเสนอแนะ

| Persona | จำนวน | ลักษณะ | ข้อเสนอแนะทางการตลาด |
|---|---|---|---|
| **VIP** | 39 (20%) | รายได้สูง (~$87k) ใช้จ่ายสูง (~82) อายุ ~33 | **กลุ่มเป้าหมายหลัก** — โปรแกรมสมาชิก/สิทธิพิเศษ สินค้าพรีเมียม รักษาไว้ให้ได้ |
| **Rich but Careful** | 35 (18%) | รายได้สูง (~$88k) แต่ใช้จ่ายต่ำ (~17) อายุ ~41 | **โอกาสเติบโตสูงสุด** — มีกำลังซื้อแต่ยังไม่ใช้จ่าย → สื่อสารเรื่องคุณภาพ/ความคุ้มค่า ข้อเสนอเฉพาะบุคคล |
| **Standard** | 81 (40%) | รายได้/ใช้จ่ายปานกลาง อายุ ~43 | กลุ่มใหญ่ที่สุด — โปรโมชันทั่วไป แต้มสะสม (แยกย่อยตามวัยได้ ดูขั้นที่ 11) |
| **Young Spender** | 22 (11%) | รายได้ต่ำ (~$26k) แต่ใช้จ่ายสูง (~79) อายุ ~25 | สินค้าแฟชั่น/เทรนด์ ผ่อนชำระ โปรโมชันนักศึกษา (ระวังความเสี่ยงด้านเครดิต) |
| **Budget Conscious** | 23 (12%) | รายได้ต่ำ (~$26k) ใช้จ่ายต่ำ (~21) อายุ ~45 | ส่วนลด สินค้าราคาประหยัด แพ็กคุ้มค่า — ไม่ควรลงงบการตลาดมาก |

> **ตัวอย่างการเขียนสรุปผลในข้อสอบ:**
> *"แบ่งลูกค้าได้ 5 กลุ่มด้วย K-Means (เลือก k = 5 จาก Elbow, Silhouette = 0.555 และ Davies-Bouldin = 0.572 ซึ่งดีที่สุด)
> ผลตรงกับ Hierarchical Clustering 98% กลุ่มที่ควรให้ความสำคัญคือ VIP (รายได้สูง-ใช้จ่ายสูง, 20%) เพื่อรักษาไว้
> และ Rich but Careful (รายได้สูง-ใช้จ่ายต่ำ, 18%) ซึ่งมีโอกาสเพิ่มยอดขายมากที่สุด"*

---
## รายการตรวจสอบ (Checklist): Clustering
- [ ] ลบคอลัมน์ ID / คอลัมน์ที่ไม่เกี่ยวกับพฤติกรรม
- [ ] **Scale ก่อนเสมอ** (ทุกวิธีใช้ระยะทาง)
- [ ] เลือก k ด้วย Elbow + Silhouette (+ Davies-Bouldin) อย่างน้อย 2 วิธี
- [ ] ทดลองอย่างน้อย 2 อัลกอริทึม แล้วเทียบ Silhouette / Davies-Bouldin
- [ ] DBSCAN: เลือก `eps` จาก k-distance plot, รายงานจำนวน Noise
- [ ] ข้อมูลเกิน 2 มิติ → ใช้ PCA ช่วยวาดกราฟ
- [ ] **ตีความแต่ละกลุ่ม** (Profile → ตั้งชื่อ Persona → ข้อเสนอแนะ) — ตัวเลขเพียงอย่างเดียวไม่เพียงพอ

## แบบฝึกหัด
1. ทดลองจัดกลุ่มด้วย `Age` + `Spending` (แบบในต้นฉบับ Kaggle) — k ที่ดีที่สุดคือเท่าใด Persona ต่างจากเดิมอย่างไร?
2. ทดลอง `AgglomerativeClustering` ด้วย `linkage='average'` และ `'complete'` แล้วเทียบ Silhouette กับ `'ward'`